# IT Service Desk Analytics — Machine Learning

## Objective
Build models to:
- Predict ticket volume
- Predict SLA breach risk

In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

In [2]:
df = pd.read_csv("../data/cleaned/cleaned_it_tickets.csv")

## Ticket Volume Prediction

In [3]:
df['created_at'] = pd.to_datetime(df['created_at'])

daily = df.groupby(df['created_at'].dt.date).size().reset_index(name='tickets')
daily['date'] = pd.to_datetime(daily['created_at'])
daily['day_of_week'] = daily['date'].dt.dayofweek
daily['month'] = daily['date'].dt.month

In [4]:
X = daily[['day_of_week', 'month']]
y = daily['tickets']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [6]:
model = RandomForestRegressor()
model.fit(X_train, y_train)

RandomForestRegressor()

In [7]:
pred = model.predict(X_test)
mean_absolute_error(y_test, pred)

3.510319768115283

## SLA Breach Prediction

In [14]:
X_cls = df[['hour', 'day_of_week']]
y_cls = df['sla_breached']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X_cls, y_cls, test_size=0.2)

In [16]:
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

RandomForestClassifier()

In [17]:
pred = clf.predict(X_test)
accuracy_score(y_test, pred)

0.9770833333333333

## Save Models for Deployment

In [18]:
joblib.dump(model, "../app/ticket_volume_model.joblib")
joblib.dump(clf, "../app/sla_model.joblib")

['../app/sla_model.joblib']

# IT Service Desk Analytics — Modelling

## Goal
Build two models:
1) Regression: Predict daily ticket volume (workforce planning)
2) Classification: Predict SLA breach risk (reduce breaches)

## Part A — Ticket Volume Prediction (Regression)
We aggregate daily ticket counts and train a model to estimate expected workload.

## Part B — SLA Breach Prediction (Classification)
We predict the probability a ticket will breach SLA using category, priority, channel, team, and time features.

## Save Models
Export trained models to the `app/` folder so Streamlit can load them.